In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

In [ ]:
import os

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

print("Project directories ready.")

Project directories ready.


In [ ]:
np.random.seed(42)
random.seed(42)

NUM_ORDERS = 5000
START_TIME = datetime(2026, 8, 26, 9, 0, 0)

SYMBOLS = ['BTCUSDT', 'ETHUSDT', 'SOLUSDT']

orders = []
fills = []
snapshots = []
incidents = []

current_time = START_TIME
risk_state = "NORMAL"
consecutive_anomalies = 0

print("Risk engine configuration loaded.")

Risk engine configuration loaded.


In [ ]:
for i in range(1, NUM_ORDERS + 1):

    order_id = f"ORD-{i:06d}"
    fill_id = f"FIL-{i:06d}"

    symbol = random.choice(SYMBOLS)
    side = random.choice(['BUY', 'SELL'])

    if symbol == 'BTCUSDT':
        base_price = 60000.0
    elif symbol == 'ETHUSDT':
        base_price = 3000.0
    else:
        base_price = 140.0

    mid_price = round(
        base_price + np.random.normal(0, base_price * 0.001),
        2
    )

    half_spread = round(mid_price * 0.0001, 2)

    best_bid = round(mid_price - half_spread, 2)
    best_ask = round(mid_price + half_spread, 2)

    # Synthetic market stress period
    is_anomaly = (
        (i % 120 >= 100)
        and
        (i % 120 <= 110)
    )

    if is_anomaly:

        bid_depth = round(random.uniform(0.1, 0.5), 4)
        ask_depth = round(random.uniform(0.1, 0.5), 4)

        latency_ms = int(
            np.random.exponential(scale=250) + 180
        )

        slippage_pct = round(
            random.uniform(0.15, 0.45),
            4
        )

        consecutive_anomalies += 1

    else:

        bid_depth = round(random.uniform(5.0, 25.0), 4)
        ask_depth = round(random.uniform(5.0, 25.0), 4)

        latency_ms = int(
            np.random.exponential(scale=12) + 5
        )

        slippage_pct = round(
            abs(np.random.normal(0.01, 0.02)),
            4
        )

        consecutive_anomalies = max(
            0,
            consecutive_anomalies - 1
        )

    requested_qty = round(
        random.uniform(0.1, 2.5),
        3
    )

    requested_price = (
        best_ask
        if side == 'BUY'
        else best_bid
    )

    price_dir = (
        1
        if side == 'BUY'
        else -1
    )

    executed_price = round(
        requested_price *
        (
            1 +
            (
                price_dir *
                (slippage_pct / 100)
            )
        ),
        2
    )

    status = (
        'REJECTED'
        if (
            is_anomaly
            and
            random.random() < 0.25
        )
        else 'FILLED'
    )

    current_time += timedelta(
        milliseconds=random.randint(50, 300)
    )

    ts_str = current_time.strftime(
        '%Y-%m-%d %H:%M:%S.%f'
    )[:-3]

    # Risk state machine

    if consecutive_anomalies >= 3:

        risk_state = "EMERGENCY_PAUSE"

    elif consecutive_anomalies == 2:

        risk_state = "THROTTLE"

    elif consecutive_anomalies == 1:

        risk_state = "WARNING"

    else:

        risk_state = "NORMAL"

    orders.append([
        order_id,
        ts_str,
        symbol,
        side,
        'MARKET',
        requested_price,
        requested_qty,
        status
    ])

    if status == 'FILLED':

        fee = round(
            executed_price *
            requested_qty *
            0.0004,
            4
        )

        fills.append([
            fill_id,
            order_id,
            ts_str,
            requested_price,
            executed_price,
            requested_qty,
            latency_ms,
            fee,
            'SUCCESS',
            slippage_pct,
            risk_state
        ])

    snapshots.append([
        ts_str,
        symbol,
        best_bid,
        best_ask,
        bid_depth,
        ask_depth
    ])

    # Incident generation

    if (
        risk_state in [
            "THROTTLE",
            "EMERGENCY_PAUSE"
        ]
        and
        is_anomaly
    ):

        incidents.append([
            f"INC-{len(incidents)+1:04d}",
            ts_str,
            (
                "CRITICAL"
                if risk_state == "EMERGENCY_PAUSE"
                else "HIGH"
            ),
            "Execution Quality",
            symbol,
            f"Slippage: {slippage_pct}%, Latency: {latency_ms}ms",
            risk_state,
            "OPEN"
        ])

print("Synthetic market simulation completed.")

Synthetic market simulation completed.


In [ ]:
orders_df = pd.DataFrame(
    orders,
    columns=[
        'order_id',
        'timestamp',
        'symbol',
        'side',
        'order_type',
        'requested_price',
        'quantity',
        'status'
    ]
)

fills_df = pd.DataFrame(
    fills,
    columns=[
        'fill_id',
        'order_id',
        'timestamp',
        'expected_price',
        'executed_price',
        'quantity',
        'latency_ms',
        'fee',
        'status',
        'slippage_pct',
        'risk_state'
    ]
)

snapshots_df = pd.DataFrame(
    snapshots,
    columns=[
        'timestamp',
        'symbol',
        'best_bid',
        'best_ask',
        'bid_depth',
        'ask_depth'
    ]
)

incidents_df = pd.DataFrame(
    incidents,
    columns=[
        'incident_id',
        'timestamp',
        'severity',
        'category',
        'symbol',
        'trigger',
        'risk_state',
        'status'
    ]
)

print("DataFrames created.")

DataFrames created.


In [ ]:
print("Orders:", len(orders_df))
print("Fills:", len(fills_df))
print("Market snapshots:", len(snapshots_df))
print("Incidents:", len(incidents_df))

Orders: 5000
Fills: 4879
Market snapshots: 5000
Incidents: 410


In [ ]:
orders_df.head()


,order_id,timestamp,symbol,side,order_type,requested_price,quantity,status
0,ORD-000001,2026-08-26 09:00:00.238,SOLUSDT,BUY,MARKET,140.08,0.636,FILLED
1,ORD-000002,2026-08-26 09:00:00.347,BTCUSDT,BUY,MARKET,59939.28,0.325,FILLED
2,ORD-000003,2026-08-26 09:00:00.504,SOLUSDT,BUY,MARKET,140.23,1.783,FILLED
3,ORD-000004,2026-08-26 09:00:00.760,BTCUSDT,SELL,MARKET,59959.15,0.116,FILLED
4,ORD-000005,2026-08-26 09:00:00.896,BTCUSDT,SELL,MARKET,60008.52,2.397,FILLED


In [ ]:
fills_df.head()

,fill_id,order_id,timestamp,expected_price,executed_price,quantity,latency_ms,fee,status,slippage_pct,risk_state
0,FIL-000001,ORD-000001,2026-08-26 09:00:00.238,140.08,140.09,0.636,20,0.0356,SUCCESS,0.0072,NORMAL
1,FIL-000002,ORD-000002,2026-08-26 09:00:00.347,59939.28,59949.11,0.325,7,7.7934,SUCCESS,0.0164,NORMAL
2,FIL-000003,ORD-000003,2026-08-26 09:00:00.504,140.23,140.27,1.783,5,0.1000,SUCCESS,0.0253,NORMAL
3,FIL-000004,ORD-000004,2026-08-26 09:00:00.760,59959.15,59958.85,0.116,7,2.7821,SUCCESS,0.0005,NORMAL
4,FIL-000005,ORD-000005,2026-08-26 09:00:00.896,60008.52,59991.54,2.397,11,57.5199,SUCCESS,0.0283,NORMAL


In [ ]:
incidents_df.head()


,incident_id,timestamp,severity,category,symbol,trigger,risk_state,status
0,INC-0001,2026-08-26 09:00:16.520,HIGH,Execution Quality,SOLUSDT,"Slippage: 0.3763%, Latency: 432ms",THROTTLE,OPEN
1,INC-0002,2026-08-26 09:00:16.635,CRITICAL,Execution Quality,SOLUSDT,"Slippage: 0.1843%, Latency: 425ms",EMERGENCY_PAUSE,OPEN
2,INC-0003,2026-08-26 09:00:16.764,CRITICAL,Execution Quality,ETHUSDT,"Slippage: 0.3655%, Latency: 354ms",EMERGENCY_PAUSE,OPEN
3,INC-0004,2026-08-26 09:00:16.864,CRITICAL,Execution Quality,BTCUSDT,"Slippage: 0.4224%, Latency: 224ms",EMERGENCY_PAUSE,OPEN
4,INC-0005,2026-08-26 09:00:17.058,CRITICAL,Execution Quality,BTCUSDT,"Slippage: 0.4051%, Latency: 198ms",EMERGENCY_PAUSE,OPEN


In [ ]:
orders_df.to_csv(
    "data/processed/orders.csv",
    index=False
)

fills_df.to_csv(
    "data/processed/fills.csv",
    index=False
)

snapshots_df.to_csv(
    "data/processed/market_snapshots.csv",
    index=False
)

incidents_df.to_csv(
    "data/processed/incidents.csv",
    index=False
)

print("CSV files successfully generated.")

CSV files successfully generated.


In [ ]:
import os

for file in os.listdir("data/processed"):
    print(file)

fills.csv
orders.csv
incidents.csv
market_snapshots.csv


In [ ]:
print("=== DATA QUALITY CHECK ===")

print("\nOrders")
print(orders_df.info())

print("\nFills")
print(fills_df.info())

print("\nIncidents")
print(incidents_df.info())

=== DATA QUALITY CHECK ===

Orders
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         5000 non-null   object 
 1   timestamp        5000 non-null   object 
 2   symbol           5000 non-null   object 
 3   side             5000 non-null   object 
 4   order_type       5000 non-null   object 
 5   requested_price  5000 non-null   float64
 6   quantity         5000 non-null   float64
 7   status           5000 non-null   object 
dtypes: float64(2), object(6)
memory usage: 312.6+ KB
None

Fills
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4879 entries, 0 to 4878
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   fill_id         4879 non-null   object 
 1   order_id        4879 non-null   object 
 2   timestamp       4879 non-null   object 
 

In [ ]:
print("Missing values - Orders")
print(orders_df.isna().sum())

print("\nMissing values - Fills")
print(fills_df.isna().sum())

print("\nMissing values - Snapshots")
print(snapshots_df.isna().sum())

print("\nMissing values - Incidents")
print(incidents_df.isna().sum())

Missing values - Orders
order_id           0
timestamp          0
symbol             0
side               0
order_type         0
requested_price    0
quantity           0
status             0
dtype: int64

Missing values - Fills
fill_id           0
order_id          0
timestamp         0
expected_price    0
executed_price    0
quantity          0
latency_ms        0
fee               0
status            0
slippage_pct      0
risk_state        0
dtype: int64

Missing values - Snapshots
timestamp    0
symbol       0
best_bid     0
best_ask     0
bid_depth    0
ask_depth    0
dtype: int64

Missing values - Incidents
incident_id    0
timestamp      0
severity       0
category       0
symbol         0
trigger        0
risk_state     0
status         0
dtype: int64


In [ ]:
total_orders = len(orders_df)

filled_orders = (
    orders_df['status'] == 'FILLED'
).sum()

rejected_orders = (
    orders_df['status'] == 'REJECTED'
).sum()

fill_rate = (
    filled_orders / total_orders * 100
)

avg_latency = fills_df['latency_ms'].mean()

avg_slippage = fills_df['slippage_pct'].mean()

open_incidents = (
    incidents_df['status'] == 'OPEN'
).sum()

print("===== EXECUTION KPIs =====")
print(f"Total Orders: {total_orders:,}")
print(f"Filled Orders: {filled_orders:,}")
print(f"Rejected Orders: {rejected_orders:,}")
print(f"Fill Rate: {fill_rate:.2f}%")
print(f"Average Latency: {avg_latency:.2f} ms")
print(f"Average Slippage: {avg_slippage:.4f}%")
print(f"Open Incidents: {open_incidents:,}")

===== EXECUTION KPIs =====
Total Orders: 5,000
Filled Orders: 4,879
Rejected Orders: 121
Fill Rate: 97.58%
Average Latency: 43.48 ms
Average Slippage: 0.0364%
Open Incidents: 410


In [ ]:
fills_df['risk_state'].value_counts()

,count
risk_state,
NORMAL,4139
EMERGENCY_PAUSE,602
WARNING,70
THROTTLE,68


In [ ]:
incidents_df['risk_state'].value_counts()

,count
risk_state,
EMERGENCY_PAUSE,369
THROTTLE,41


In [ ]:
from google.colab import files

files.download(
    "data/processed/orders.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download(
    "data/processed/fills.csv"
)

files.download(
    "data/processed/market_snapshots.csv"
)

files.download(
    "data/processed/incidents.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>